This is another implementation of RAG-based applications for information retrieval and summarization of texts using LLMs. As opposed to RAG_policy_file.ipynb, we are using LangChain, which offers a high-level API that makes it unnecessary to manually tokenize, embed, index and search te files.

## Load libraries

In [10]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory


from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes, DecodingMethods

# Custom funcs
import llm_funcs.langchain_funcs as langchain_helpers


## Preprocess text file to search

In [2]:
# Create text loader and lod document
loader = TextLoader('source_files/companyPolicies.txt')
documents = loader.load()

# Create text splitter and split into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0, separator='\n')
texts = text_splitter.split_documents(documents)

## Embed and store text for later search and retrieval

In [3]:
embeddings = HuggingFaceEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings)  # store the embedding in docsearch using Chromadb

/Users/manfernandez/anaconda3/envs/langchain/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8074.01it/s]


## Construct LLM model

In [ ]:
# Specify model parameters
params = {
    "decoding_method": DecodingMethods.GREEDY,
    "max_new_tokens": 256,
    "min_new_tokens": 130,
    "temperature": 0.5,
}

model_id = "mistralai/mistral-small-3-1-24b-instruct-2503"

mistral_small_llm = langchain_helpers.llm_model(model_id, params)

## Create applications using LangChain

In [5]:
# Simple Q&A Chatbot
qa = RetrievalQA.from_chain_type(llm=mistral_small_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is mobile policy?"
qa.invoke(query)

{'query': 'what is mobile policy?',
 'result': ' The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance. It covers acceptable use, security, confidentiality, cost management, compliance, handling of lost or stolen devices, and consequences of non-compliance. The policy is regularly reviewed to align with evolving technology and security best practices. The policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices.'}

In [6]:
# Summarizing app
query = "Can you summarize the document for me?"
qa.invoke(query)

{'query': 'Can you summarize the document for me?',
 'result': ' The documents outline the fundamental principles and ethical standards that guide the organization, including a commitment to integrity, respect, accountability, safety, and environmental responsibility. The Code of Conduct emphasizes acting honestly, embracing diversity, taking responsibility for actions, prioritizing safety, and promoting sustainable practices. The Recruitment Policy focuses on attracting diverse candidates, maintaining transparency, and ensuring objective selection criteria. The Anti-discrimination and Harassment Policy prohibits any form of discrimination or harassment, promoting a respectful and inclusive workplace. Additionally, the organization prioritizes health and safety, complying with relevant laws and regulations to maintain a hazard-free environment. The documents collectively emphasize the importance of ethical conduct, diversity, inclusion, and safety within the organization.'}

#### Fine-tuning the model's behaviour

In [7]:
# Define custom prompt
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

# Convert into template
PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [9]:
# Generate Q&A
qa = RetrievalQA.from_chain_type(llm=mistral_small_llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 chain_type_kwargs=chain_type_kwargs, # Automatically maps retrieved passages to {context} and query to {question}
                                 return_source_documents=False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "I don't know. The document does not provide information about eating in company vehicles. It only discusses the smoking policy for company vehicles. If you have any other questions related to the provided document, feel free to ask. If you have questions about eating in company vehicles, you should ask your employer. I am only able to answer questions based on the information provided in the document. I am not able to make up answers. If I don't know the answer, I will say that I don't know. I will not try to make up an answer. I will only answer questions based on the information provided in the document. I will not answer questions that are not related to the information provided in the document."}

#### Including memory in the LLM

In [14]:
# Generate Q&A application with memory support
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

qa = ConversationalRetrievalChain.from_llm(llm=mistral_small_llm, 
                                           chain_type="stuff", 
                                           retriever=docsearch.as_retriever(), 
                                           memory = memory, 
                                           get_chat_history=lambda h : h, 
                                           return_source_documents=False)

In [17]:
# Now, every time I ask something to the model, I can store it in the memory and pass it later to give extra context
# Initialize empty memory
history = []

query = "What is mobile policy?"
result = qa.invoke({"question":query}, {"chat_history": history}) # Invoke result considering memory
print(result["answer"])

# Append to running memory
history.append((query, result["answer"]))

 The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance. Key points include:

- **Acceptable Use**: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.
- **Security**: Safeguard your mobile device and access credentials. Exercise caution when downloading apps or clicking links from unfamiliar sources. Promptly report security concerns or suspicious activities related to your mobile device.
- **Confidentiality**: Avoid transmitting sensitive company information via unsecured messaging apps or emails. Be discreet when discussing company matters in public spaces.
- **Cost Management**: Keep personal phone usage separate from company accounts and reimburse the comp

In [19]:
# Ask new things
query = "List points in it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The key points in the mobile policy are:

1. **Cost Management**: Keep personal phone usage separate from company accounts and reimburse the company for any personal charges on company-issued phones.
2. **Compliance**: Adhere to all pertinent laws and regulations concerning mobile phone usage, including those related to data protection and privacy.
3. **Lost or Stolen Devices**: Immediately report any lost or stolen mobile devices to the IT department or your supervisor.
4. **Consequences**: Non-compliance with this policy may lead to disciplinary actions, including the potential loss of mobile phone privileges.
5. **Acceptable Use**: Mobile devices are primarily intended for work-related tasks. Limited personal usage is allowed, provided it does not disrupt work obligations.
6. **Security**: Safeguard your mobile device and access credentials. Exercise caution when downloading apps or clicking links from unfamiliar sources. Promptly report security concerns or suspicious activities r

In [20]:
# Repeat recursively
history.append((query, result["answer"]))
query = "What is the aim of it?"
result = qa({"question": query}, {"chat_history": history})
print(result["answer"])

 The aim of the Mobile Phone Policy is to promote the responsible and secure use of mobile devices in line with legal and ethical standards. The policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent with company values and legal compliance. Every employee is expected to comprehend and abide by these guidelines. Regular reviews of the policy ensure its ongoing alignment with evolving technology and security best practices. The Mobile Phone Policy is aimed at promoting the responsible and secure use of mobile devices in line with legal and ethical standards. The Mobile Phone Policy sets forth the standards and expectations governing the appropriate and responsible usage of mobile devices in the organization. The purpose of this policy is to ensure that employees utilize mobile phones in a manner consistent wi